# embed_test - choose the embedding model
Muhammad Ahmad, CMS 542432, BSCS-15-D. Runtime > Change runtime type > GPU (T4). Research prototype, not legal advice.
Upload `embed_units.json` and `tests/retrieval_test.json` (the reviewed file). Metrics are computed on PARENT sections: a hit means the correct section is among the top-k distinct sections.

In [ ]:
# Muhammad Ahmad, CMS 542432, BSCS-15-D
# Qwen3-Embedding needs a recent transformers (model card: >=4.51.0) and sentence-transformers (>=2.7.0); verify on the model card.
!pip -q install -U sentence-transformers transformers

In [ ]:
# Muhammad Ahmad, CMS 542432, BSCS-15-D
from google.colab import files
up = files.upload()          # select embed_units.json and retrieval_test.json
print(list(up))

In [ ]:
# Muhammad Ahmad, CMS 542432, BSCS-15-D
# Only needed for EmbeddingGemma (gated model: accept the licence on its Hugging Face page first). Skip otherwise.
# from huggingface_hub import login; login()

In [ ]:
# Muhammad Ahmad, CMS 542432, BSCS-15-D
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
output_dir = Path("/content/drive/MyDrive/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Muhammad Ahmad, CMS 542432, BSCS-15-D
import gc, json, time
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
units = json.load(open("embed_units.json", encoding="utf-8"))
tests = json.load(open("retrieval_test.json", encoding="utf-8"))
print(len(units), "units;", len(tests), "test questions")

QWEN_TASK = "Given a Pakistani legal question, retrieve the article or section of law that answers it"
MODELS = {   # q = how a QUERY is formatted, d = how a DOCUMENT is formatted (prefixes differ per model!)
    "qwen3-0.6b": dict(id="Qwen/Qwen3-Embedding-0.6B", q=lambda s: f"Instruct: {QWEN_TASK}\nQuery:{s}", d=lambda s: s),
    "embeddinggemma-300m": dict(id="google/embeddinggemma-300m", q=lambda s: f"task: search result | query: {s}",
                                d=lambda s: f"title: none | text: {s}"),
    "bge-small-en-v1.5": dict(id="BAAI/bge-small-en-v1.5",
                              q=lambda s: "Represent this sentence for searching relevant passages: " + s, d=lambda s: s),
}
parent_of = [u["parent_id"] for u in units]

def evaluate(name, cfg):
    model = SentenceTransformer(cfg["id"], device="cuda" if torch.cuda.is_available() else "cpu")
    model.max_seq_length = 512
    D = model.encode([cfg["d"](u["embed_text"]) for u in units], batch_size=32, normalize_embeddings=True, show_progress_bar=True)
    Q = model.encode([cfg["q"](t["question"]) for t in tests], batch_size=32, normalize_embeddings=True, show_progress_bar=True)
    S = Q @ D.T
    ranks = []
    for qi, t in enumerate(tests):
        seen = {}
        for j in np.argsort(-S[qi]):
            seen.setdefault(parent_of[j], len(seen) + 1)      # rank of each distinct parent section
        ranks.append(seen.get(t["source_id"], 10**6))
    r = np.array(ranks)
    return {"model": name, "recall@1": 100*(r<=1).mean(), "recall@3": 100*(r<=3).mean(),
            "recall@5": 100*(r<=5).mean(), "MRR": float(np.mean(1.0 / r))}

rows, failures = [], []
for name, cfg in tqdm(MODELS.items(), desc="Embedding models", unit="model"):
    started = time.perf_counter()
    try:
        result = evaluate(name, cfg)
        with open(output_dir / f"embed_results_{name}.json", "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        rows.append(result)
        print(f"{name}: finished in {time.perf_counter() - started:.1f}s; saved to Drive")
    except Exception as e:
        failures.append((name, str(e)))
        print(f"{name}: FAILED after {time.perf_counter() - started:.1f}s: {e}")
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

columns = ["model", "recall@1", "recall@3", "recall@5", "MRR"]
df = pd.DataFrame(rows, columns=columns).round(3)
print("\nEmbedding results:")
print(df.to_string(index=False) if rows else "No model succeeded.")
if failures:
    print("\nFAILED MODELS:")
    for name, error in failures:
        print(f"- {name}: {error}")
df.to_csv("embed_results.csv", index=False)

Mount Drive before the evaluation cell. Every completed model is checkpointed under `MyDrive/outputs/`.

Pick the model with the best recall@5 and MRR, then set `rp.CFG['EMBED_KEY']` in `run_rag.ipynb` (keys: qwen3-0.6b, embeddinggemma-300m, bge-small-en-v1.5). Report all three rows in the slides.